In [1]:
import pickle
import sys
import copy
import time

import cobra

import multiprocessing
import multiprocessing.pool
# from multiprocessing import Process
# from threading import Thread

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

In [2]:
# import pandas as pd
# build_files_path = '/data2/hratch/human_me/build_files/'
# human_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/toy_model.json')
# full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
# full_model_public = cobra.io.read_sbml_model(lp_path + 'recon2_2.xml')

# required_metabolites = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)



# You are here

In [3]:
from tqdm import tqdm

In [154]:
lp_path = '/data2/hratch/human_me/test_lp/'
error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]', '40s_rrna_protein_complex[c]', 
                     '60s_rrna_protein_complex[c]']

def remove_metabolite(test_metabolites = []):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    
    # test
    me_metabolites += error_metabolites

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    me_model.add_reactions(ra)
    _, status, __ = me_model.solve_lp(mu_val = 0.01)
    return status.max()

In [147]:
remove_metabolite()

Getting MINOS parameters...
Done in 291.812 seconds with status 1


1

In [243]:
lp_path = '/data2/hratch/human_me/test_lp/'
error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]', '40s_rrna_protein_complex[c]', 
                     '60s_rrna_protein_complex[c]']
    
def remove_metabolite_2(test_metabolites = []):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = required_metabolites['n']
#     me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    # test
#     me_metabolites += error_metabolites + required_metabolites['n']
    me_metabolites = sorted(set(me_metabolites))

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    me_model.add_reactions(ra)
    sln, status, __ = me_model.solve_lp(mu_val = 0.01)
    return sln, status.max()

In [244]:
sln, status = remove_metabolite_2()

Getting MINOS parameters...
Done in 72.8979 seconds with status 1


In [241]:
test = [m for m in me_model.reactions if 'HGNC:17821' in m.id]

In [252]:
idx = 0
print(test[idx].id)
print(sln[me_model.reactions.index(test[idx].id)])

HGNC:17821_TRANSCRIPTION_ELONGATION
-1.6875317657953238e-17


In [248]:
test[idx].id

'HGNC:17821_lariats_DEGRADATIONn_0'

In [225]:
test[6]

Reaction identifier,HGNC:17821_TRANSLATION_ELONGATIONc
Name,
Memory address,0x07f3191fe9208
Stoichiometry,mu/(mu + 0.02) HGNC:17821_mrna[c] + 0.0693147180559945/(mu + 0.02) HGNC:17821_mrna_deg_proxy + 2.93209936320823e6*mu 5.86419872641646e8 TRANSLATION_ELONGATIONc_protein_complex_complex[c] + 49.5286... mu/(mu + 0.02) + 0.0693147180559945/(mu + 0.02) + 2.93209936320823e6*mu 5.86419872641646e8 + 49.52861063999808 + 22 + 9 + 30 + 36 + 25 + 18 + 10 + 23 + 28 + 40 + 15 + 14 + 18 + 26...
GPR,HGNC:3189 and HGNC:3214 and HGNC:3208 and HGNC:3300 and ribosome
Lower bound,0.0
Upper bound,1000.0


In [ ]:
# mrna[n] requirement: no flux through transcription elongation and transcription processing!!
# mrna_deg_proxy requirement: no flux through transcription degradation reaction
# additionally, no flux through protein degradation (polyub reaction, deubiquitination, or proteosome) reaction; 
# 2 lariat degradation reactions created

In [178]:
0 flux through elongation reaction...

In [179]:
test

['utp[n]', 'gtp[n]', 'ctp[n]', 'atp[n]', '2_protein_complex[n]']

In [ ]:
# 2 lariat degradation reactions